In [ ]:
import pandas as pd
from src.preprocessing import load_raw_population_excel, load_ssdse_data, build_neighbor_list, build_spatial_lag
from src.features import calculate_ssdse_indicators, run_pca_analysis
from src.models import fit_ols_model, forecast_future
from src.visualization import make_level_map, load_ibaraki_shapefile

# 1. データ構築
df_pop = load_raw_population_excel("data/population_ibaraki.xlsx")
df_ssdse = load_ssdse_data("data/SSDSE-E-2024.csv")
# 指標の算出と結合
df_ssdse = calculate_ssdse_indicators(df_ssdse)
panel = pd.merge(df_pop, df_ssdse, on=["city_name", "year"], how="left")

# 2. 空間関係の構築
cities = panel["city_name"].unique().tolist()
neigh_df = build_neighbor_list("data/N03-20240101_08.shp", cities)
neigh_dict = neigh_df.groupby("city_name")["neighbor_name"].apply(list).to_dict()
panel = build_spatial_lag(panel, neigh_dict)

# 3. モデル推定と将来予測（RTX 4070 Ti のパワーをここで活用）
features = ["log_pop_lag1", "W_log_pop_lag1", "share_over65", "soc_inc_rate"]
model = fit_ols_model(panel, "log_pop", features)
forecast_df = forecast_future(panel, neigh_dict, model, features[2:], [2030, 2040])

# 4. 可視化
gdf = load_ibaraki_shapefile("data/N03-20240101_08.shp")
make_level_map(forecast_df, gdf, 2040)